# Sawti-ASR — reconnaissance vocale de la Darija marocaine

**Point d'étape, fin de semaine 1.** Projet d'initiation 4A, filière IA, ENSAM Meknès.

Ce notebook est exécuté de bout en bout sur Kaggle (CPU) : les tableaux, graphiques et
extraits audio ci-dessous sont produits par le code du dépôt
[sawti-asr](https://github.com/Mohamedaminebelasri/sawti-asr), branche `main`.

**Plan :** 1 · Contexte et objectifs — 2 · Pipeline — 3 · Données — 4 · Traitement — 5 · Splits —
6 · Exemples à écouter — 7 · Résultats E0 — 8 · État de l'art et prochaines étapes.

## 1. Contexte et objectif

**Le problème.** La Darija marocaine est très peu dotée en reconnaissance vocale. Trois difficultés
s'ajoutent :

- **la Darija elle-même** : un dialecte sans orthographe fixée, éloigné de l'arabe standard sur
  lequel les modèles comme Whisper ont surtout appris ;
- **le code-switching** : on passe sans cesse du Darija au français dans la même phrase
  (« غادي نمشي للـ réunion ») ;
- **le bruit** : rue, café, voiture… les enregistrements réels sont rarement propres.

**L'approche.** Partir de Whisper (OpenAI), l'affiner (fine-tuning complet et LoRA) sur de la Darija,
avec de l'**augmentation audio** (bruit, vitesse, pitch, SpecAugment), et tout suivre dans MLflow.

**Objectifs chiffrés** (gains relatifs de WER, mesurés sur nos jeux de test) :

| Jeu de test | Objectif |
|---|---|
| Bruité (SNR 20 à 0 dB) | gain **≥ 10 %** |
| Code-switching | gain **≥ 5 %** |
| Propre | perte **≤ 1 point** de WER |

Le WER (*word error rate*) compte les mots substitués, supprimés et insérés, divisés par le nombre
de mots de la référence : plus il est bas, mieux c'est. Le CER fait la même chose sur les caractères.

### Mise en place (technique)

Cette cellule prépare l'environnement Kaggle : elle lit les jetons (Hugging Face, MLflow) depuis un
dataset Kaggle **privé**, sans jamais afficher leur valeur, clone le dépôt et installe ses dépendances.
Elle n'a rien à montrer à l'encadrant, mais tout le reste en dépend.

In [ ]:
import glob, os, subprocess, sys

# 1) Jetons : dataset Kaggle privé "sawti-secrets" (lignes NOM=valeur). Aucune valeur n'est affichée.
SECRETS = ["HF_TOKEN", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]
fichiers = sorted(glob.glob("/kaggle/input/**/secrets.env", recursive=True))
lus = {}
if fichiers:
    with open(fichiers[0], encoding="utf-8-sig") as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne and not ligne.startswith("#") and "=" in ligne:
                nom, valeur = ligne.split("=", 1)
                lus[nom.strip()] = valeur.strip().strip('"').strip("'")
for nom in SECRETS:
    if lus.get(nom):
        os.environ[nom] = lus[nom]
    print(f"{nom}: {'OK' if lus.get(nom) else 'ABSENT'}")
del lus
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ.get("HF_TOKEN", "")

# 2) Dépôt + dépendances + ESC-50
DEPOT = "/kaggle/temp/sawti-asr"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(cmd)
    return r.stdout

sh(f"rm -rf {DEPOT} && git clone -q --depth 1 https://github.com/Mohamedaminebelasri/sawti-asr {DEPOT}")
print(sh(f"git -C {DEPOT} log -1 --format='Code du dépôt : commit %h (%s)'"))
sh(f"cd {DEPOT} && {sys.executable} -m pip install -q -r requirements.txt && {sys.executable} -m pip install -q -e .")
sh(f"cd {DEPOT} && git clone -q --depth 1 https://github.com/karolpiczak/ESC-50 data/ESC-50")
os.chdir(DEPOT)
sys.path.insert(0, os.path.join(DEPOT, "src"))
print("Prêt.")

## 2. Architecture du pipeline

Le schéma ci-dessous résume la chaîne complète. En **vert**, ce qui est fait (semaine 1) ; en **gris**,
ce qui vient ensuite. Chaque étape est un script du dépôt, et chaque expérience est enregistrée
dans MLflow (serveur DagsHub) : paramètres, métriques et prédictions.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fait, avenir = "#cfe8d5", "#e6e6e6"
# (colonne, ligne, texte, couleur) : 4 colonnes, 3 lignes, aucune flèche ne croise une case
etapes = {  # « | » = retour à la ligne dans la case
    "src":  (0, 2, "Sources|DODa · MoulSot · Casablanca", fait),
    "prep": (1, 2, "prepare_data.py|16 kHz · 0,3–30 s|normalisation · filtres", fait),
    "hub":  (2, 2, "Hub HF (privé)|rechargé par fetch_data.py", fait),
    "test": (3, 2, "8 jeux de test|propre · 4 bruités (fold 5)|MoulSot · CS · Casablanca", fait),
    "esc":  (0, 1, "ESC-50|folds 1–4 : augmentation|fold 5 : tests bruités", fait),
    "aug":  (1, 1, "Augmentation (train)|bruit · vitesse · pitch|SpecAugment", avenir),
    "ft":   (2, 1, "Fine-tuning Whisper|full FT · LoRA|(E1, E2…)", avenir),
    "eval": (3, 1, "Évaluation|E0 zero-shot : fait|E1, E2 : à venir", fait),
    "mlf":  (3, 0, "MLflow (DagsHub)|métriques · prédictions|registre de modèles", fait),
}
W, H, DX, DY = 2.3, 1.15, 3.0, 1.75
pos = lambda k: (etapes[k][0] * DX, etapes[k][1] * DY)
fig, ax = plt.subplots(figsize=(13, 6.2))
for k, (cx, cy, t, c) in etapes.items():
    x, y = pos(k)
    ax.add_patch(FancyBboxPatch((x - W / 2, y - H / 2), W, H, boxstyle="round,pad=0.05", fc=c, ec="#555", lw=1.2))
    ax.text(x, y, t.replace("|", chr(10)), ha="center", va="center", fontsize=9.5)
def fleche(a, b, texte=None):
    (xa, ya), (xb, yb) = pos(a), pos(b)
    if ya == yb:   # horizontale
        p0, p1 = (xa + W / 2 + 0.05, ya), (xb - W / 2 - 0.05, yb)
    else:          # verticale
        s = 1 if yb > ya else -1
        p0, p1 = (xa, ya + s * (H / 2 + 0.05)), (xb, yb - s * (H / 2 + 0.05))
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="->", color="#333", lw=1.4))
    if texte:
        ax.text((p0[0] + p1[0]) / 2 + 0.08, (p0[1] + p1[1]) / 2, texte, fontsize=8, color="#444", va="center")
for a, b, t in [("src", "prep", None), ("prep", "hub", None), ("hub", "test", None),
                ("esc", "aug", None), ("aug", "ft", None), ("ft", "eval", None),
                ("hub", "ft", "train + validation"), ("test", "eval", None), ("eval", "mlf", None)]:
    fleche(a, b, t)
ax.set_xlim(-1.4, 3 * DX + 1.4); ax.set_ylim(-0.8, 2 * DY + 0.8); ax.axis("off")
ax.set_title("Pipeline Sawti-ASR (vert : fait · gris : semaines 2 à 4)", fontsize=12)
plt.show()

## 3. Sources de données

Quatre sources, chacune avec un rôle précis. Le tableau est construit à partir de la configuration
du dépôt (`configs/data.yaml`), pour qu'il reste à jour.

- **DODa** : phrases **lues**, propres, 7 locuteurs identifiés → entraînement et **test propre**.
- **MoulSot** : parole **spontanée** issue de YouTube, avec beaucoup de code-switching ; les
  transcriptions ont été produites par Gemini (pas par des humains) → entraînement (≈ 20 h retenues)
  et deux tests (spontané, code-switching).
- **Casablanca** : **test externe** uniquement, jamais vu à l'entraînement. Attention : le français
  y est translittéré en alphabet arabe, ce qui ne suit pas notre convention.
- **ESC-50** : 2 000 bruits d'environnement (50 catégories, 5 « folds »). Les folds 1 à 4 servent à
  l'augmentation, le fold 5 uniquement aux tests bruités : **aucun bruit de test n'est vu à
  l'entraînement**.

Les licences comptent : Casablanca (CC BY-NC-ND) et ESC-50 (CC BY-NC) interdisent l'usage commercial,
d'où un dataset préparé **privé** sur le Hub et aucun audio dans le dépôt Git.

In [ ]:
import pandas as pd
from sawti_asr.config import load_config

cfg = load_config("configs/base.yaml", "configs/data.yaml")
sources = pd.DataFrame([
    ["DODa", cfg["doda"]["repo"], "lu, propre, 7 locuteurs", "train + validation + test propre", "MIT"],
    ["MoulSot", cfg["moulsot"]["repo"], "spontané, YouTube, code-switching",
     f"train (≈{cfg['moulsot']['max_train_hours']} h) + test spontané + test CS", "non déclarée"],
    ["Casablanca", f"{cfg['casablanca']['repo']} ({cfg['casablanca']['config']})", "spontané, séries TV",
     "test externe uniquement", "CC BY-NC-ND 4.0"],
    ["ESC-50", "karolpiczak/ESC-50", "bruits d'environnement",
     f"augmentation (folds 1–4) · tests bruités (fold {cfg['noise']['test_fold']})", "CC BY-NC 3.0"],
], columns=["Source", "Dépôt", "Type de parole", "Rôle dans le projet", "Licence"])
sources.style.hide(axis="index")

## 4. Traitement des données

Toutes les sources passent par le même traitement (`scripts/prepare_data.py`) :

1. **Rééchantillonnage à 16 kHz, mono** : c'est le format attendu par Whisper.
2. **Filtre de durée : 0,3 à 30 s.** Whisper lit des fenêtres de 30 s ; un extrait plus long
   serait tronqué alors que sa transcription resterait complète. Les extraits trop courts sont du bruit.
3. **Normalisation du texte**, appliquée de la même façon aux références et aux prédictions avant
   de calculer le WER : retrait des diacritiques (harakat) et du tatweel, unification des alef
   (أ إ آ → ا) et des trois écritures du « g » (ڭ گ → ݣ), retrait de la ponctuation,
   minuscules pour le latin.
4. **Convention d'écriture** : Darija en **alphabet arabe**, mots français en **latin**,
   **pas d'Arabizi** (Darija écrite en lettres latines, « kidayr labas »).
5. **Filtre `drop_latin_only`** : un extrait dont la référence ne contient **aucune lettre arabe**
   (Arabizi ou autre langue) est retiré du train et de la validation, ainsi que de `test_moulsot`.
   Une référence en Arabizi ne peut pas être comparée à une sortie en alphabet arabe : le WER
   mesurerait l'écriture, pas la reconnaissance.

La cellule suivante montre la normalisation sur 3 exemples, puis la décision du filtre.

In [ ]:
from sawti_asr.data.normalize import normalize_text, has_arabic, has_latin, is_code_switching

print(f"Fréquence : {cfg['sampling_rate']} Hz · durée gardée : {cfg['min_duration_s']} à {cfg['max_duration_s']} s")
exemples = [
    "واشْ نْتَا مْزْيَانْ؟ أَنَا لاباس!",          # diacritiques + alef + ponctuation
    "غـــادي نمشي للـ Réunion ديال الصباح.",    # tatweel + latin en minuscules
    "ڭال ليا: گاع الناس جاو",                   # trois écritures du « g »
]
pd.DataFrame({"texte brut": exemples,
              "texte normalisé": [normalize_text(t, **cfg["text"]) for t in exemples]})

Et voici la décision du filtre `drop_latin_only` : seuls les textes **sans aucune lettre arabe**
sont retirés. Le code-switching (arabe + latin) est **gardé** : c'est justement ce qu'on veut
apprendre à reconnaître.

In [ ]:
textes = ["كيداير لاباس", "غادي نمشي للـ réunion", "kidayr labas cv", "on se voit demain"]
pd.DataFrame({
    "texte": textes,
    "lettres arabes": [has_arabic(t) for t in textes],
    "lettres latines": [has_latin(t) for t in textes],
    "code-switching": [is_code_switching(t) for t in textes],
    "décision": ["gardé" if has_arabic(t) else "retiré (sans arabe)" for t in textes],
})

## 5. Les splits

Les données préparées (≈ 4 Go) sont stockées une seule fois sur le Hugging Face Hub, dans un dataset
**privé**. Au début de chaque session, `scripts/fetch_data.py` les recharge (quelques minutes au lieu
de 13 min de préparation), puis reconstruit les 4 jeux de test bruités avec une graine fixe :
ils sont identiques à chaque session.

In [ ]:
print(sh(f"{sys.executable} scripts/fetch_data.py 2>&1 | grep -v -E 'it/s|examples/s|%\\|' | tail -5"))

Le tableau donne, pour chaque split, le nombre d'extraits, les heures d'audio et la répartition par
source. On ne lit que les colonnes de texte et de durée : l'audio n'est pas décodé, c'est rapide.

In [ ]:
from datasets import load_from_disk

dd = load_from_disk("data/processed")
noisy = load_from_disk("data/test_noisy")
meta = {k: dd[k].select_columns(["text", "source", "speaker", "duration"]).to_pandas() for k in dd}

lignes = []
for k, df in meta.items():
    lignes.append({"split": k, "extraits": len(df), "heures": round(df.duration.sum() / 3600, 2),
                   "durée moyenne (s)": round(df.duration.mean(), 1),
                   "sources": ", ".join(f"{s} {n}" for s, n in df.source.value_counts().items())})
for k in noisy:
    lignes.append({"split": f"test_noisy_{k}", "extraits": len(noisy[k]),
                   "heures": round(lignes[[l["split"] for l in lignes].index("test_clean_doda")]["heures"], 2),
                   "durée moyenne (s)": round(meta["test_clean_doda"].duration.mean(), 1),
                   "sources": "doda (test propre) + bruit ESC-50 fold 5"})
ordre = ["train", "validation", "test_clean_doda", "test_noisy_snr_20", "test_noisy_snr_10",
         "test_noisy_snr_5", "test_noisy_snr_0", "test_moulsot", "test_cs_moulsot", "test_casablanca_all"]
tableau = pd.DataFrame(lignes).set_index("split").reindex([o for o in ordre if o in {l["split"] for l in lignes}])
tableau

**Distribution des durées.** DODa contient des phrases courtes et lues (2 à 3 s), MoulSot et
Casablanca des extraits spontanés plus longs. Tous restent sous la limite de 30 s de Whisper.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for src, c in [("doda", "#4c72b0"), ("moulsot", "#dd8452")]:
    d = meta["train"].query("source == @src").duration
    axes[0].hist(d, bins=60, range=(0, 30), alpha=0.7, color=c, label=f"{src} ({len(d)} extraits)")
axes[0].set(title="Train : durées par source", xlabel="durée (s)", ylabel="extraits"); axes[0].legend()
tests = ["test_clean_doda", "test_moulsot", "test_cs_moulsot", "test_casablanca_all"]
axes[1].boxplot([meta[t].duration for t in tests], showfliers=False)
axes[1].set_xticks(range(1, len(tests) + 1), [t.replace("test_", "") for t in tests])
axes[1].set(title="Jeux de test : durées (sans valeurs extrêmes)", ylabel="durée (s)")
plt.tight_layout(); plt.show()

**Arabe seul ou code-switching ?** Pour chaque split, la part des extraits écrits uniquement en
alphabet arabe et la part qui mélange arabe et latin (code-switching). Après le filtre, il ne reste
plus d'extrait « latin seul » dans le train, la validation et `test_moulsot`.

In [ ]:
def ecriture(t):
    return "code-switching" if is_code_switching(t) else ("arabe seul" if has_arabic(t) else "latin seul")

parts = pd.DataFrame({k: meta[k].text.map(ecriture).value_counts(normalize=True) * 100
                      for k in ["train", "validation"] + tests}).T.fillna(0)
parts = parts.reindex(columns=[c for c in ["arabe seul", "code-switching", "latin seul"] if c in parts])
ax = parts.plot.barh(stacked=True, figsize=(12, 4), color=["#4c72b0", "#dd8452", "#c44e52"][:parts.shape[1]])
ax.set(xlabel="% des extraits", title="Écriture des références par split"); ax.invert_yaxis()
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.18), ncol=3, frameon=False)
for i, v in enumerate(parts.get("code-switching", pd.Series(0, index=parts.index))):
    ax.text(101, i, f"CS : {v:.1f} %", va="center", fontsize=9)
ax.set_xlim(0, 115); plt.tight_layout(); plt.show()
parts.round(1)

**Pourquoi découper par locuteur et par chaîne ?** Si une même voix (ou une même chaîne YouTube)
apparaît à la fois dans le train et dans le test, le modèle peut « reconnaître la voix » plutôt que
la langue, et le score de test est trop optimiste. On sépare donc :

- **DODa par locuteur** : certains locuteurs sont réservés au test, un autre à la validation ;
- **MoulSot par chaîne** : les chaînes du test officiel sont exclues du train, et environ 1 chaîne
  sur 20 est réservée à la validation.

La cellule vérifie qu'il n'y a **aucun recouvrement** entre le train et les tests.

In [ ]:
d = cfg["doda"]
print("DODa · locuteurs de test :", d["test_speakers"], "· de validation :", d["val_speakers"])
def locuteurs(split, src):
    return set(meta[split].query("source == @src").speaker)
for src, tests_src in [("doda", ["test_clean_doda"]), ("moulsot", ["test_moulsot", "test_cs_moulsot"])]:
    tr = locuteurs("train", src)
    for t in tests_src:
        te = locuteurs(t, src)
        mot = "locuteurs" if src == "doda" else "chaînes"
        print(f"{src:8s} {t:16s} : {len(tr)} {mot} en train, {len(te)} en test, recouvrement = {len(tr & te)}")

## 6. Exemples à écouter

D'abord une phrase **propre** de DODa (locuteur jamais vu à l'entraînement), puis **la même phrase**
mélangée à un bruit ESC-50 du fold de test à 20, 10, 5 puis 0 dB. À 0 dB, la parole et le bruit ont
la même énergie.

In [ ]:
from IPython.display import Audio, display, Markdown

propre = dd["test_clean_doda"]
i = int(next(j for j, x in enumerate(meta["test_clean_doda"].duration) if 2.5 <= x <= 4.5))
ex = propre[i]
display(Markdown(f"**Propre** — « {ex['text']} » ({ex['duration']:.1f} s, locuteur {ex['speaker']})"))
display(Audio(ex["audio"]["array"], rate=16000))
for k in ["snr_20", "snr_10", "snr_5", "snr_0"]:
    nx = noisy[k][i]
    assert nx["text"] == ex["text"]  # même phrase
    display(Markdown(f"**{k.replace('snr_', 'SNR ')} dB** — bruit : *{nx['noise_category']}*"))
    display(Audio(nx["audio"]["array"], rate=16000))

Puis 2 extraits du test **code-switching** (MoulSot) : de la parole spontanée où le français
apparaît en latin dans la transcription.

In [ ]:
cs = dd["test_cs_moulsot"]
choix = [j for j, x in enumerate(meta["test_cs_moulsot"].duration) if 3 <= x <= 8][:2]
for j in choix:
    ex = cs[j]
    display(Markdown(f"**Code-switching** — « {ex['text']} » ({ex['duration']:.1f} s)"))
    display(Audio(ex["audio"]["array"], rate=16000))

## 7. Résultats E0 : Whisper sans entraînement

E0 est la **référence de départ** : `openai/whisper-base` (74 M de paramètres) utilisé tel quel,
langue forcée à « ar », sur tous les jeux de test (GPU T4, ≈ 21 min). Les résultats sont lus
directement dans le run MLflow enregistré sur DagsHub.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

RUN_ID = "c2bd2145fb794c4fae42b9c2e4ba41f3"
LIEN = f"https://dagshub.com/Mohamedaminebelasri/sawti-asr.mlflow/#/experiments/0/runs/{RUN_ID}"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
run = MlflowClient().get_run(RUN_ID)
m = run.data.metrics
jeux = ["test_clean_doda", "test_noisy_snr_20", "test_noisy_snr_10", "test_noisy_snr_5",
        "test_noisy_snr_0", "test_moulsot", "test_cs_moulsot", "test_casablanca_all"]
e0 = pd.DataFrame([{"jeu": j, "WER": m.get(f"{j}/wer"), "CER": m.get(f"{j}/cer"),
                    "temps (s)": m.get(f"{j}/seconds")} for j in jeux]).set_index("jeu")
display(Markdown(f"Run **{run.info.run_name}** · modèle `{run.data.tags.get('model')}` · "
                 f"[ouvrir le run sur DagsHub]({LIEN})"))
e0.round(3)

**Le WER selon le bruit.** Plus le SNR baisse (plus le bruit est fort), plus le WER monte, de façon
régulière. C'est exactement ce que l'augmentation par le bruit doit corriger.

In [ ]:
x = ["propre", "20 dB", "10 dB", "5 dB", "0 dB"]
wer = e0.loc[jeux[:5], "WER"].values; cer = e0.loc[jeux[:5], "CER"].values
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, wer, "o-", label="WER", color="#c44e52"); ax.plot(x, cer, "s--", label="CER", color="#4c72b0")
for xi, w in zip(x, wer):
    ax.annotate(f"{w:.2f}", (xi, w), textcoords="offset points", xytext=(0, 8), ha="center")
ax.axhline(1.0, color="#999", lw=0.8, ls=":")
ax.set(title="E0 whisper-base : erreur selon le niveau de bruit (test DODa)", ylabel="taux d'erreur",
       xlabel="niveau de bruit"); ax.legend(); plt.tight_layout(); plt.show()

**Pourquoi un WER supérieur à 1 ?** Le WER compte aussi les mots **insérés**. Quand Whisper
« hallucine » (répète un mot en boucle ou invente une suite), il ajoute plus de mots que la
référence n'en contient. La cellule télécharge les prédictions enregistrées dans le run et
décompose les erreurs en substitutions, suppressions et insertions.

In [ ]:
import jiwer
dossier = mlflow.artifacts.download_artifacts(run_id=RUN_ID, artifact_path="predictions")
def preds(jeu):
    df = pd.read_csv(f"{dossier}/whisper-base_{jeu}_predictions.csv").fillna("")
    df["ref"] = df.reference.map(lambda t: normalize_text(t, **cfg["text"]))
    df["hyp"] = df.prediction.map(lambda t: normalize_text(t, **cfg["text"]))
    return df[df.ref != ""]
lignes = []
for jeu in ["test_clean_doda", "test_cs_moulsot", "test_moulsot"]:
    df = preds(jeu)
    o = jiwer.process_words(list(df.ref), list(df.hyp))
    n = o.hits + o.substitutions + o.deletions
    longues = (df.hyp.str.split().str.len() > 3 * df.ref.str.split().str.len()).sum()
    lignes.append({"jeu": jeu, "mots de référence": n,
                   "substitutions": f"{o.substitutions / n:.0%}", "suppressions": f"{o.deletions / n:.0%}",
                   "insertions": f"{o.insertions / n:.0%}", "prédictions 3× trop longues": int(longues)})
pd.DataFrame(lignes).set_index("jeu")

**Trois erreurs typiques.**

1. **Hallucination** : Whisper répète un mot en boucle jusqu'à la limite de longueur.
2. **Français translittéré** : avec la langue forcée à « ar », le mot français est écrit en
   lettres arabes ; la référence le garde en latin, donc le mot est compté faux.
3. **Hallucination sur une phrase propre courte** : même sur du DODa propre, la sortie déborde.

In [ ]:
cs_p, propre_p = preds("test_cs_moulsot"), preds("test_clean_doda")
ratio = lambda df: df.hyp.str.split().str.len() / df.ref.str.split().str.len().clip(lower=1)
h = cs_p.loc[ratio(cs_p).idxmax()]
avec_fr = cs_p[cs_p.reference.str.contains("transport")]
if avec_fr.empty:
    avec_fr = cs_p[cs_p.ref.map(has_latin) & ~cs_p.hyp.map(has_latin)]
t = avec_fr.iloc[0]
p = propre_p.loc[ratio(propre_p).idxmax()]
couper = lambda s, n=220: s if len(s) <= n else s[:n] + " …"
erreurs = pd.DataFrame([
    ["1. hallucination (CS)", h.reference, couper(h.prediction)],
    ["2. français translittéré (CS)", t.reference, couper(t.prediction)],
    ["3. hallucination (propre)", p.reference, couper(p.prediction)],
], columns=["type", "référence", "prédiction de Whisper"]).set_index("type")
pd.set_option("display.max_colwidth", 260)
erreurs

**À retenir pour E0.** Whisper-base sans entraînement ne sait pas transcrire la Darija : le WER
dépasse 1 partout. La plupart des mots sont mal reconnus (substitutions), et les insertions dues
aux hallucinations font passer le WER au-dessus de 1. Le français est translittéré au lieu d'être
gardé en latin. Ce n'est pas un échec : c'est la **ligne de base** que le fine-tuning
et l'augmentation doivent faire baisser. Une partie des hallucinations pourra aussi être limitée au
décodage (pénalité de répétition), à tester séparément.

## 8. État de l'art et prochaines étapes

Deux modèles récents servent de repère. Leurs chiffres viennent de leurs cartes de modèle sur
Hugging Face : ils utilisent **leur propre normalisation et leur propre jeu de test**, et ne sont
donc **pas directement comparables** à nos WER (à mesurer nous-mêmes avec notre normalisation si
besoin).

| Modèle | Base | Données | WER / CER annoncés | Remarques |
|---|---|---|---|---|
| [MoulSot v0.3](https://huggingface.co/atlasia/moulsot.v0.3) (AtlasIA, mai 2026) | Qwen3-ASR 1,7 B, affiné | ≈ 1 500 h collectées sur 51 chaînes YouTube, dont 80 h transcrites par Gemini 2.5 Pro (le dataset MoulSot que nous utilisons) | 38,99 % / 12,58 % | Apache-2.0 ; français gardé en latin, comme notre convention ; mieux que ISMA (39,20 %) |
| [VoxaPlus](https://huggingface.co/Nmaghura/VoxaPlus) | modèle compact ≈ 60 M de paramètres, distillé d'un modèle 1,7 B | audio marocain avec transcriptions assistées par IA | 36,87 % (phase 4, test Darija du projet) | annoncé à ≈ 2,5 points de son « professeur » 28× plus gros ; chiffres repris de la carte du modèle, non revérifiés |

Notre point de départ (E0, whisper-base 74 M) est très loin de ces modèles. L'objectif du projet
n'est pas de les battre, mais de **mesurer proprement l'apport de l'augmentation** sur le bruit et
le code-switching, avec un pipeline reproductible.

**Prochaines étapes (proposition, à valider avec l'encadrant) :**

| Semaine | Travail |
|---|---|
| 2 | Module d'augmentation (bruit ESC-50 folds 1–4, vitesse, pitch, SpecAugment) — écrit et testé, en relecture ; script d'entraînement Whisper ; **E1** : fine-tuning **sans** augmentation |
| 3 | **E2** : fine-tuning **avec** augmentation ; comparaison full fine-tuning / LoRA ; réglage du décodage contre les hallucinations |
| 4 | Évaluation finale sur les 8 jeux de test, vérification des objectifs (≥ 10 % bruité, ≥ 5 % CS, ≤ 1 point sur le propre), registre MLflow du meilleur modèle, rapport |